---
execute:
  skip: true
authors:
- name: Steffen Bollmann
doi: 10.5281/zenodo.18717009
github: https://github.com/neurodesk/neurodeskedu/blob/main/books/examples/workflows/nextflow_neurodesk.ipynb
edit_url: https://github.dev/neurodesk/neurodeskedu/blob/main/books/examples/workflows/nextflow_neurodesk.ipynb
downloads:
- url: /edu/_sources/examples/workflows/nextflow_neurodesk.ipynb
  title: Download source notebook
  filename: nextflow_neurodesk.ipynb
  static: false
---
# Nextflow in Neurodesk

:::{admonition} Under review
:class: nd-review-badge nd-review-badge--in-progress

[review issue](<https://github.com/neurodesk/neurodeskedu-reviews/issues/33>)
:::

:::{dropdown} Run this notebook
:class: nd-launch

- [Run in Australia](<https://play.neurodesk.cloud.edu.au/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/workflows/nextflow_neurodesk.ipynb&branch=main>)
- [Run in Europe](<https://play-europe.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/workflows/nextflow_neurodesk.ipynb&branch=main>)
- [Run in America](<https://play-america.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/workflows/nextflow_neurodesk.ipynb&branch=main>)
- [Run in Global Server 1](<https://play-europe.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/workflows/nextflow_neurodesk.ipynb&branch=main>)
- [Run in Global Server 2](<https://edu.neurodesk.org/hub/user-redirect/git-pull?repo=https%3A//github.com/neurodesk/neurodeskedu&urlpath=lab/tree/neurodeskedu/books/examples/workflows/nextflow_neurodesk.ipynb&branch=main>)
:::



**Date:** 13 Feb 2026 (last updated Nov 2026)

**License:** 
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>


**Note:** If this notebook uses neuroimaging tools from Neurocontainers, those tools retain their original licenses. Please see <a href="https://neurodesk.org/overview/how-to-cite-us/" target="_blank" style="color: #0066cc;">Neurodesk citation guidelines</a> for details.

<details style="border-left: 4px solid #f0a500; background-color: rgba(240, 165, 0, 0.08); border-radius: 4px; padding: 0.75em 1em; margin: 1em 0;">
  <summary style="cursor: pointer; display: flex; justify-content: space-between; align-items: center; list-style: none; font-size: 1.1em;">
    <strong style="color: black;">✨ Use of AI</strong>
    <span style="color: black !important; font-size: 1.1em;">▽</span>
  </summary>
  <p style="font-size: 1.1em;">The revisions made during peer review were prepared with assistance from <em>Anthropic's Claude (via Claude Code)</em>: Claude identified possible issues, and the reviewer (Ashley Stewart) checked them and included the corrections they agreed with. The author reviewed the final content and takes responsibility for it.</p>
</details>

### Citation and Resources
#### Workflow engine
- Di Tommaso, P., Chatzou, M., Floden, E. W., Barja, P. P., Palumbo, E., & Notredame, C. (2017). Nextflow enables reproducible computational workflows. *Nature Biotechnology, 35*(4), 316-319. [https://doi.org/10.1038/nbt.3820](https://doi.org/10.1038/nbt.3820)

#### Tools included in this workflow
__FSL - Brain Extraction Tool (BET)__
- M. Jenkinson, C.F. Beckmann, T.E. Behrens, M.W. Woolrich, S.M. Smith. FSL. NeuroImage, 62(2), 782-790, 2012. [https://doi.org/10.1016/j.neuroimage.2011.09.015](https://doi.org/10.1016/j.neuroimage.2011.09.015)
- Smith S. M. (2002). Fast robust automated brain extraction. Human brain mapping, 17(3), 143–155. [https://doi.org/10.1002/hbm.10062](https://doi.org/10.1002/hbm.10062)

#### Dataset from OSF
- Shaw, T., & Bollmann, S. (2020). Dataset for Towards Optimising MRI Methods for ChAracterisation of Tissue (TOMCAT) [Data set]. OSF. [https://doi.org/10.17605/OSF.IO/BT4EZ](https://doi.org/10.17605/OSF.IO/BT4EZ)

## Table of contents
1. [Introduction](#introduction)
2. [Load software tools and import python libraries](#load-software-tools-and-import-python-libraries)
3. [Download test data](#download-test-data)
4. [Hello Nextflow](#hello-nextflow)
5. [Core concepts](#core-concepts)
6. [Single-subject brain extraction](#single-subject-brain-extraction)
7. [Multi-subject pipeline](#multi-subject-pipeline)
8. [Visualize results](#visualize-results)
9. [Next steps](#next-steps)
10. [Cleanup](#cleanup)
11. [Dependencies and environment capture](#dependencies-and-environment-capture)

## Introduction

[Nextflow](https://www.nextflow.io/) is a workflow engine that lets you write data-driven computational pipelines. It handles parallelism, file staging, and error recovery so you can focus on the analysis logic.

**Why use Nextflow for neuroimaging?**
- Automatically runs independent subjects in parallel
- Tracks which steps have finished so you can resume failed runs with `-resume`
- Works the same way on your laptop, an HPC cluster, or in the cloud
- Large ecosystem of ready-made pipelines at [nf-core](https://nf-co.re/)

This notebook teaches Nextflow from scratch through three progressively complex examples:
1. A "hello world" pipeline
2. Brain extraction on a single subject
3. A multi-subject pipeline with quality-control summary

## Load software tools and import python libraries

Load FSL via the Neurodesk module system. Loading a module puts the tool on the `PATH` of this notebook, and Nextflow's default local executor runs each task as a child process that inherits that `PATH`. That is why the `bet` and `fslstats` calls inside the pipelines below work without any extra configuration.

On an HPC cluster (for example with the SLURM executor), tasks run on other nodes in a fresh environment, so you would load the tool per process with the `module` directive (e.g. `module 'fsl/6.0.7.16'`), a `beforeScript`, or run it in a container with the `container` directive.

In [13]:
import module
await module.load('fsl/6.0.7.16')
await module.list()

['fsl/6.0.7.16']

In [ ]:
#nextflow is installed inside Neurodesk
!nextflow -version

## Download test data

We download a T1-weighted MRI scan from the [TOMCAT dataset](https://doi.org/10.17605/OSF.IO/BT4EZ) (OSF) through its [HuggingFace mirror](https://huggingface.co/datasets/neurodeskorg/tomcat-sample), then create a copy as a simulated second subject for the multi-subject demo later.

In [ ]:
%%bash
mkdir -p data

if [ -f ./data/sub-01.nii ]; then
    echo "sub-01.nii already exists, skipping download"
else
    if [ ! -f ./data/sub-01.nii.gz ]; then
        echo "Downloading T1w from the HuggingFace mirror of OSF bt4ez..."
        curl -fL --retry 5 --retry-all-errors -o ./data/sub-01.nii.gz \
            https://huggingface.co/datasets/neurodeskorg/tomcat-sample/resolve/c97be23c616dc57da291c5d3dfa711991b2212cc/TOMCAT_DIB/sub-01/ses-01_7T/anat/sub-01_ses-01_7T_T1w_defaced.nii.gz
    fi
    echo "Decompressing..."
    gunzip ./data/sub-01.nii.gz
fi

echo "Done."
ls -lh ./data/sub-01.nii

In [ ]:
%%bash
# Create a simulated second subject by copying sub-01
if [ ! -f ./data/sub-02.nii ]; then
    cp ./data/sub-01.nii ./data/sub-02.nii
    echo "Created sub-02.nii"
else
    echo "sub-02.nii already exists"
fi

ls -lh ./data/

## Hello Nextflow

Every Nextflow pipeline is a `.nf` script containing **processes** (units of work) and a **workflow** that wires them together using **channels** (asynchronous data queues).

Let's start with the simplest possible pipeline.

In [ ]:
%%writefile hello.nf
process SAY_HELLO {
    input:
    val greeting

    output:
    stdout

    script:
    """
    echo "$greeting from Nextflow!"
    """
}

workflow {
    greetings = channel.of('Hello', 'Bonjour', 'Hola')
    SAY_HELLO(greetings) | view
}

In [ ]:
!nextflow run hello.nf

**What just happened?**

1. `channel.of(...)` created a channel with three items
2. Nextflow launched the `SAY_HELLO` process **three times** — once per item, potentially in parallel. Run the cell again and the greetings may come back in a different order: tasks finish in whatever order they complete
3. Each execution ran in its own isolated `work/` subdirectory

Let's peek inside the work directory to see how Nextflow organises task execution:

In [ ]:
%%bash
echo "=== work directory structure ==="
# Show the first task directory as an example
TASK_DIR=$(find work -name '.command.sh' -print -quit 2>/dev/null | xargs dirname)
if [ -n "$TASK_DIR" ]; then
    echo "Example task dir: $TASK_DIR"
    echo ""
    echo "--- .command.sh (the actual script that ran) ---"
    cat "$TASK_DIR/.command.sh"
    echo ""
    echo "--- .command.log (stdout/stderr) ---"
    cat "$TASK_DIR/.command.log"
else
    echo "No work directory found (pipeline may not have run)"
fi

## Core concepts

| Concept | Description |
|---------|-------------|
| **Process** | A unit of work with defined inputs, outputs, and a script. Runs in an isolated directory. |
| **Channel** | An asynchronous queue that connects processes. Data flows through channels. |
| **Workflow** | The top-level block that creates channels and wires processes together. |
| **`publishDir`** | Copies output files from the work directory to a permanent results folder. |
| **`params`** | Pipeline parameters that can be set on the command line with `--name value`. |

Nextflow automatically handles:
- **Parallelism**: If a channel has N items, the process runs N times (potentially concurrently)
- **File staging**: Input files are symlinked into each task's work directory
- **Resumability**: Use `-resume` to skip already-completed tasks after a failure

## Single-subject brain extraction

Now let's do something useful: run FSL's `bet` (Brain Extraction Tool) on a single T1w image via Nextflow.

This introduces:
- `path` inputs (file handling)
- `params` for configurable settings
- `publishDir` to save outputs to a results folder

In [ ]:
%%writefile bet_single.nf
params.input = './data/sub-01.nii'
params.outdir = './results_single'
params.frac  = 0.4

process BET {
    publishDir params.outdir, mode: 'copy'

    input:
    path t1w

    output:
    path '*_brain*.nii.gz'

    script:
    """
    bet ${t1w} ${t1w.baseName}_brain -f ${params.frac} -m
    """
}

workflow {
    input_ch = channel.fromPath(params.input)
    BET(input_ch)
}

In [ ]:
!nextflow run bet_single.nf

In [ ]:
!ls -lh results_single/

**Key points:**
- `channel.fromPath(...)` creates a channel from a file path
- Inside the script block, `${t1w}` refers to the staged input file
- `publishDir` copies the outputs matching `'*_brain*.nii.gz'` (the brain image and, because of `-m`, the brain mask) to our results folder
- We could override any parameter from the command line, e.g. `nextflow run bet_single.nf --frac 0.3`

## Multi-subject pipeline

Real neuroimaging studies have multiple subjects. Nextflow makes this easy — we just put multiple files into a channel and Nextflow fans out automatically.

This pipeline has two processes:
1. **BET** — runs brain extraction per subject (parallel fan-out)
2. **QC_SUMMARY** — collects all results and generates a summary table (runs once after all BET tasks finish)

In [ ]:
%%writefile bet_multi.nf
params.inputs = './data/sub-*.nii'
params.outdir = './results_multi'
params.frac   = 0.4

process BET {
    publishDir "${params.outdir}/bet", mode: 'copy'

    input:
    path t1w

    output:
    path '*_brain.nii.gz', emit: brain
    path '*_brain_mask.nii.gz', emit: mask

    script:
    """
    bet ${t1w} ${t1w.baseName}_brain -f ${params.frac} -m
    """
}

process QC_SUMMARY {
    publishDir params.outdir, mode: 'copy'

    input:
    path mask_images

    output:
    path 'qc_summary.tsv'

    script:
    """
    echo -e "subject\tbrain_voxels\tbrain_volume_mm3" > qc_summary.tsv
    for mask in *_brain_mask.nii.gz; do
        subj=\$(echo \$mask | sed 's/_brain_mask.nii.gz//')
        # fslstats -V prints the number of voxels and the volume in mm^3
        read nvox vol <<< \$(fslstats \$mask -V)
        echo -e "\${subj}\t\${nvox}\t\${vol}" >> qc_summary.tsv
    done
    echo "=== QC Summary ==="
    cat qc_summary.tsv
    """
}

workflow {
    t1w_ch = channel.fromPath(params.inputs)

    BET(t1w_ch)

    QC_SUMMARY(BET.out.mask.collect())
}

In [ ]:
!nextflow run bet_multi.nf

In [ ]:
%%bash
echo "=== Output files ==="
ls -lh results_multi/bet/
echo ""
echo "=== QC Summary ==="
cat results_multi/qc_summary.tsv

**What's new here?**

- `channel.fromPath('data/sub-*.nii')` picks up both `sub-01.nii` and `sub-02.nii`
- Nextflow runs `BET` twice in parallel (one per subject)
- `emit:` names each output, so the workflow can refer to `BET.out.mask` instead of `BET.out[1]`
- `.collect()` gathers all per-subject outputs into a single list and passes it to `QC_SUMMARY`
- `QC_SUMMARY` runs **once**, after all BET tasks complete, and generates a combined table. The two rows are identical because sub-02 is a copy of sub-01

This fan-out/collect pattern is the foundation of most neuroimaging Nextflow pipelines.

### Resuming a pipeline

Nextflow caches every task it has finished. Run the same pipeline again with `-resume` and the tasks whose inputs and script have not changed are reported as `cached` instead of being run again. This is what lets you restart a large study after a failure without redoing the subjects that already finished.

In [ ]:
!nextflow run bet_multi.nf -resume

Changing a parameter invalidates the cache for the affected tasks. Try `!nextflow run bet_multi.nf -resume --frac 0.3`: both `BET` tasks and `QC_SUMMARY` run again, because the `bet` command changed.

## Visualize results

Use ipyniivue to check the brain extraction: the original T1w is shown in grey with the brain mask from `bet` overlaid in red. Brain tissue outside the red region was cut off, and red outside the brain is non-brain tissue that was kept.

In [ ]:
from ipyniivue import NiiVue

nv = NiiVue()
nv.load_volumes([
    {"path": "./data/sub-01.nii", "colormap": "gray"},
    {"path": "./results_multi/bet/sub-01_brain_mask.nii.gz", "colormap": "red", "opacity": 0.5}
])
nv

## Next steps
You now know the core Nextflow patterns. Here are some ways to extend what you've learned:

- **`-resume`**: Add this flag to skip already-completed tasks when re-running a pipeline after a failure or parameter change
- **`nextflow.config`**: Move parameters, executor settings (local/SLURM/PBS), and resource limits (CPUs, memory) into a separate config file
- **Containers**: Nextflow can pull and run Docker/Singularity containers per process — set `container` in a process or config
- **nf-core**: Browse [nf-co.re](https://nf-co.re/) for production-grade neuroimaging pipelines and community best practices
- **More modalities**: Extend the glob pattern (`params.inputs`) to pick up T2w, FLAIR, or functional data

## Cleanup

Nextflow keeps every task's files in `work/` (for `-resume`) and writes its logs to `.nextflow.log*`. These can grow large in real studies, so delete them once you no longer need to resume. The published results in `results_single/` and `results_multi/` are copies and are kept.

In [ ]:
%%bash
rm -rf work .nextflow .nextflow.log*
ls

## Dependencies and environment capture
- Using the package [watermark](https://github.com/rasbt/watermark) to document system environment and software versions used in this notebook, alongside the Neurodesktop version extracted from the `JUPYTER_IMAGE` or `NEURODESKTOP_VERSION` environment variables.

In [1]:
import os

%load_ext watermark

%watermark
%watermark --iversions

neurodesktop_version = (
    os.environ.get('JUPYTER_IMAGE', '').split(':')[-1] or
    os.environ.get('NEURODESKTOP_VERSION', 'unknown')
)

print(f"Neurodesktop version: {neurodesktop_version}")

# Versions of the tools used in the pipelines
print(await module.list())
!nextflow -version

Last updated: 2026-03-22T23:47:03.970654+00:00

Python implementation: CPython
Python version       : 3.13.9
IPython version      : 9.7.0

Compiler    : GCC 14.3.0
OS          : Linux
Release     : 5.15.0-164-generic
Machine     : x86_64
Processor   : x86_64
CPU cores   : 32
Architecture: 64bit

ipyniivue: 2.4.4

Neurodesktop version: 2025-12-20
